In [49]:
import os
from collections import Counter
from eval_functions import *

In [50]:
def count_operators(text):
    operators = ['AND', 'OR', 'NOT']
    counts = defaultdict(int)
    positions = {op: [] for op in operators}

    words = text.split()
    for i, word in enumerate(words):
        clean_word = word.strip('[],()')  # Entfernt Klammern und Kommas
        if clean_word in operators:
            counts[clean_word] += 1
            positions[clean_word].append(i)

    return counts, positions

def compare_operators(label_counts, label_positions, model_counts, model_positions):
    comparison = {
        'AND': {'count_match': label_counts['AND'] == model_counts['AND'], 'position_match': label_positions['AND'] == model_positions['AND']},
        'OR': {'count_match': label_counts['OR'] == model_counts['OR'], 'position_match': label_positions['OR'] == model_positions['OR']},
        'NOT': {'count_match': label_counts['NOT'] == model_counts['NOT'], 'position_match': label_positions['NOT'] == model_positions['NOT']}
    }
    return comparison


In [54]:
main_directory = 'model_output'
models = find_folders_with_output(main_directory)
eval_path = f"metrics"
os.makedirs(eval_path, exist_ok=True)
results = []



In [55]:
for model in [models[0]]:# models:
    p2_label_path = "../../chia_label/p1_model_input"
    ready_path = f"model_output/{model}/output"

    label_files = {extract_nct_number(f): os.path.join(p2_label_path, f) for f in os.listdir(p2_label_path) if f.endswith('.txt')}
    model_files = {extract_nct_number(f): os.path.join(ready_path, f) for f in os.listdir(ready_path) if f.endswith('.txt')}
    common_ncts = set(label_files.keys()).intersection(model_files.keys())

    for nct in common_ncts:
        label_data = read_text(label_files[nct])
        model_data = read_text(model_files[nct])
        label_counts, label_positions = count_operators(label_data)
        model_counts, model_positions = count_operators(model_data)

        comparison = compare_operators(label_counts, label_positions, model_counts, model_positions)
        bleu_score = calculate_bleu(reference=label_data, hypothesis=model_data)
        jaccard_score = jaccard_similarity(label_data, model_data)



        print("-" * 40)
        print(f"NCT: {nct}")
        print(f"Label counts: {dict(label_counts)}")  # Gibt den Zähler als Dictionary aus
        print(f"Model counts: {dict(model_counts)}")
    
        print(label_data)
        print("-" * 20)
        print(model_data)
        print(f"Comparison: {comparison}")
        print(f"BLEU Score: {bleu_score}")
        print(f"Jaccard Similarity: {jaccard_score}")
        print("-" * 40)

        result = {
            'NCT': nct,
            'Model': model,
            'Label AND': label_counts['AND'],
            'Label OR': label_counts['OR'],
            'Label NOT': label_counts['NOT'],
            'Model AND': model_counts['AND'],
            'Model OR': model_counts['OR'],
            'Model NOT': model_counts['NOT'],
            'BLEU Score': bleu_score,
            'Jaccard Similarity': jaccard_score,
            'AND Count Match': comparison['AND']['count_match'],
            'AND Position Match': comparison['AND']['position_match'],
            'OR Count Match': comparison['OR']['count_match'],
            'OR Position Match': comparison['OR']['position_match'],
            'NOT Count Match': comparison['NOT']['count_match'],
            'NOT Position Match': comparison['NOT']['position_match'],
        }
        results.append(result)

df_results = pd.DataFrame(results)

In [56]:
df_results

### Model Output to nice JSON and Failure 

In [ ]:
for model in models:
    p2_label_path = "../../chia_label/p1_model_input"
    ready_path = f"model_output/{model}/output"
    
    label_files = {extract_nct_number(f): os.path.join(p2_label_path, f) for f in os.listdir(p2_label_path) if f.endswith('.txt')}
    model_files = {extract_nct_number(f): os.path.join(ready_path, f) for f in os.listdir(ready_path) if f.endswith('.txt')}
    common_ncts = set(label_files.keys()).intersection(model_files.keys())
    labels = []
    predictions = []
    success_data = []
    
    for nct in common_ncts:
    
        label_data = read_text(label_files[nct])
        model_data = read_text(model_files[nct])
            
        
        bleu_score = calculate_bleu(reference=label_data, hypothesis=model_data)
        jaccard_score = jaccard_similarity(label_data, model_data)
  
        print(f"BLEU Score: {bleu_score}")
        print(f"Jaccard Similarity: {jaccard_score}")
        



    
